In [16]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re
import xml.etree.ElementTree as ET

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns, clean_text
import xml.etree.ElementTree as ET

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_DIR = SILVER_DIR / "boe_candidates_docs_text"

BOE_CANDIDATES_DOCS_TEXT_PATH = BOE_CANDIDATES_DOCS_TEXT_DIR / "boe_candidates_docs_text.parquet"

In [17]:
REQUIRED_XML_DOWNLOAD_COLS = {"identificador", "doc_file_stem", "url_xml"}

REQUIRED_DOCS_TEXT_COLS = [
    "identificador",
    "doc_file_stem",
    "url_html",
    "url_xml",
    "fecha_publicacion",
    "titulo",
    "epigrafe_nombre",
    "departamento_nombre",
    "seccion_nombre",
    "xml_path",
    "texto_limpio",
    "texto_len",
    "parsed_at",
]

In [18]:
test = pd.read_parquet(
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
)

test.columns

Index(['identificador', 'doc_file_stem', 'titulo', 'fecha_publicacion', 'year',
       'month', 'day', 'url_html', 'url_xml', 'url_pdf', 'pdf_size_bytes',
       'pdf_size_kbytes', 'pagina_inicial', 'pagina_final', 'diario_numero',
       'seccion_codigo', 'seccion_nombre', 'departamento_codigo',
       'departamento_nombre', 'epigrafe_nombre', 'control', 'item_location',
       'source', 'country', 'bronze_date', 'seccion_nombre_norm',
       'departamento_nombre_norm', 'epigrafe_nombre_norm', 'titulo_norm'],
      dtype='object')

# Funciones

Código incremental. Se añaden nuevas filas y las ya procesadas ok se quedan tal cual.

In [19]:
def parse_boe_xml_to_text(xml_path: Path) -> str:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    text_parts = []

    for elem in root.iter():
        if elem.text and elem.text.strip():
            text_parts.append(elem.text.strip())

    return clean_text(" ".join(text_parts))

In [20]:
def get_existing_ids(output_path: Path) -> set[str]:
    if not output_path.exists():
        return set()

    existing = pd.read_parquet(output_path)

    if "identificador" not in existing.columns:
        raise ValueError("El parquet existente no contiene la columna 'identificador'.")

    return set(existing["identificador"].dropna().astype(str))

In [21]:
def build_boe_candidates_docs_text(
    candidates: pd.DataFrame,
    xml_dir: Path,
    output_path: Path,
) -> pd.DataFrame:
    """
    Construye o actualiza la capa silver `boe_candidates_docs_text`.

    Conserva todos los candidatos BOE y añade el texto parseado del XML
    cuando está disponible.

    Estados posibles:
    - ok: XML encontrado y parseado correctamente.
    - missing: XML no encontrado en disco.
    - parse_error: XML encontrado, pero no se pudo parsear.

    La función es incremental:
    - no reprocesa registros con xml_status == "ok";
    - reprocesa registros nuevos;
    - reprocesa registros previously missing o parse_error.
    """
    required_cols = {
        "identificador",
        "doc_file_stem",
        "url_html",
        "url_xml",
        "fecha_publicacion",
        "titulo",
        "epigrafe_nombre",
        "departamento_nombre",
        "seccion_nombre",
    }

    output_cols = [
        "identificador",
        "doc_file_stem",
        "url_html",
        "url_xml",
        "fecha_publicacion",
        "titulo",
        "epigrafe_nombre",
        "departamento_nombre",
        "seccion_nombre",
        "xml_path",
        "texto_limpio",
        "texto_len",
        "xml_status",
        "parse_error",
        "parsed_at",
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    if not xml_dir.exists():
        raise FileNotFoundError(f"El directorio de XML no existe: {xml_dir}")

    validate_required_columns(candidates, required_cols)

    candidates = candidates.copy()
    candidates["identificador"] = candidates["identificador"].astype(str)
    candidates["doc_file_stem"] = candidates["doc_file_stem"].astype(str)

    if output_path.exists():
        existing_df = pd.read_parquet(output_path)

        if "identificador" not in existing_df.columns:
            raise ValueError(
                "El parquet existente no contiene la columna 'identificador'."
            )

        if "xml_status" not in existing_df.columns:
            existing_df["xml_status"] = "unknown"

        ok_ids = set(
            existing_df.loc[
                existing_df["xml_status"].eq("ok"),
                "identificador",
            ].astype(str)
        )
    else:
        existing_df = pd.DataFrame(columns=output_cols)
        ok_ids = set()

    candidates_to_process = candidates[
        ~candidates["identificador"].isin(ok_ids)
    ].copy()

    if candidates_to_process.empty:
        final_df = existing_df.copy()

        for col in output_cols:
            if col not in final_df.columns:
                final_df[col] = None

        final_df = (
            final_df[output_cols]
            .drop_duplicates(subset=["identificador"], keep="last")
            .sort_values(["fecha_publicacion", "identificador"])
            .reset_index(drop=True)
        )

        final_df.to_parquet(output_path, index=False)
        return final_df

    records = []
    parsed_at = datetime.now(timezone.utc).isoformat()

    for row in candidates_to_process.itertuples(index=False):
        xml_path = xml_dir / f"{row.doc_file_stem}.xml"

        base_record = {
            "identificador": row.identificador,
            "doc_file_stem": row.doc_file_stem,
            "url_html": row.url_html,
            "url_xml": row.url_xml,
            "fecha_publicacion": row.fecha_publicacion,
            "titulo": row.titulo,
            "epigrafe_nombre": row.epigrafe_nombre,
            "departamento_nombre": row.departamento_nombre,
            "seccion_nombre": row.seccion_nombre,
            "xml_path": str(xml_path),
            "parsed_at": parsed_at,
        }

        if not xml_path.exists():
            records.append(
                {
                    **base_record,
                    "texto_limpio": "",
                    "texto_len": 0,
                    "xml_status": "missing",
                    "parse_error": "XML file not found",
                }
            )
            continue

        try:
            texto_limpio = parse_boe_xml_to_text(xml_path)

            records.append(
                {
                    **base_record,
                    "texto_limpio": texto_limpio,
                    "texto_len": len(texto_limpio),
                    "xml_status": "ok",
                    "parse_error": None,
                }
            )

        except Exception as exc:
            records.append(
                {
                    **base_record,
                    "texto_limpio": "",
                    "texto_len": 0,
                    "xml_status": "parse_error",
                    "parse_error": repr(exc),
                }
            )

    new_df = pd.DataFrame.from_records(records, columns=output_cols)

    final_df = pd.concat([existing_df, new_df], ignore_index=True)

    for col in output_cols:
        if col not in final_df.columns:
            final_df[col] = None

    final_df = (
        final_df[output_cols]
        .drop_duplicates(subset=["identificador"], keep="last")
        .sort_values(["fecha_publicacion", "identificador"])
        .reset_index(drop=True)
    )

    final_df.to_parquet(output_path, index=False)

    return final_df

# Pruebas

In [22]:
boe_candidates = pd.read_parquet(BOE_CANDIDATES_PATH)
boe_candidates.loc[boe_candidates["identificador"]=="BOE-A-2024-16664"]

,identificador,doc_file_stem,titulo,fecha_publicacion,year,month,day,url_html,url_xml,url_pdf,...,epigrafe_nombre,control,item_location,source,country,bronze_date,seccion_nombre_norm,departamento_nombre_norm,epigrafe_nombre_norm,titulo_norm
165,BOE-A-2024-16664,20240810_BOE-A-2024-16664,"Resolución de 22 de julio de 2024, de la Direc...",2024-08-10,2024,8,10,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,https://www.boe.es/boe/dias/2024/08/10/pdfs/BO...,...,Instalaciones eléctricas,2024/12772,departamento.epigrafe.item,boe,ES,20240810,iii otras disposiciones,ministerio para la transicion ecologica y el r...,instalaciones electricas,resolucion de 22 de julio de 2024 de la direcc...


In [23]:
boe_candidates = pd.read_parquet(BOE_CANDIDATES_PATH)

boe_candidates_docs_text = build_boe_candidates_docs_text(
    candidates=boe_candidates,
    xml_dir=BOE_DOCS_XML_DIR,
    output_path=BOE_CANDIDATES_DOCS_TEXT_PATH,
)

In [24]:
boe_candidates_docs_text["xml_status"].value_counts(dropna=False)

xml_status
ok    1131
Name: count, dtype: int64

In [25]:
boe_candidates_docs_text["texto_limpio"][0]

'BOE-B-2021-32554 Ministerio de Transportes, Movilidad y Agenda Urbana Resolución de la Dirección General de Planificación y Evaluación de la Red Ferroviaria por la que se abre Información Pública correspondiente al Expediente de Expropiación Forzosa 305ADIF2104 motivado por las obras del "Proyecto de Construcción para la Implantación del Ancho Estándar en el Corredor Mediterráneo. Tramo: Castellbisbal-Murcia. Subtramo: Vinaroz-Vandellós. Vía y Electrificación", en los términos municipales de Camarles, Freginals, L’Aldea, L’Ametlla de Mar y Ulldecona (Tarragona) y Vinarós (Castellón). Boletín Oficial del Estado 20210707 161 5 B A210042578 43051 43053 https://www.boe.es/boe/dias/2021/07/07/pdfs/BOE-B-2021-32554.pdf B ADIF-Alta Velocidad, el 12 de mayo de 2021 insta la incoación del expediente de expropiación forzosa para disponer de los terrenos necesarios para la ejecución de las obras del Proyecto de referencia, aprobado el 4 de abril de 2019. Dicha aprobación conlleva la declaración 

In [26]:
boe_candidates_docs_text.columns

Index(['identificador', 'doc_file_stem', 'url_html', 'url_xml',
       'fecha_publicacion', 'titulo', 'epigrafe_nombre', 'departamento_nombre',
       'seccion_nombre', 'xml_path', 'texto_limpio', 'texto_len', 'xml_status',
       'parse_error', 'parsed_at'],
      dtype='object')

In [27]:
boe_candidates_docs_text.loc[
    boe_candidates_docs_text["identificador"].isin([
        "BOE-B-2021-32560",
        "BOE-A-2023-2598",
        "BOE-A-2023-10306",
        "BOE-B-2023-19082",
        "BOE-A-2024-16664",
    ])
]


,identificador,doc_file_stem,url_html,url_xml,fecha_publicacion,titulo,epigrafe_nombre,departamento_nombre,seccion_nombre,xml_path,texto_limpio,texto_len,xml_status,parse_error,parsed_at
6,BOE-B-2021-32560,20210707_BOE-B-2021-32560,https://www.boe.es/diario_boe/txt.php?id=BOE-B...,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,2021-07-07,Anuncio del Área de Industria y Energía de la ...,None,MINISTERIO DE POLÍTICA TERRITORIAL Y FUNCIÓN P...,V. Anuncios. - B. Otros anuncios oficiales,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-B-2021-32560 Ministerio de Política Territ...,8323,ok,None,2026-06-18T11:07:32.082516+00:00
65,BOE-A-2023-2598,20230131_BOE-A-2023-2598,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...",Impacto ambiental,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-A-2023-2598 Estatal Ministerio para la Tra...,101414,ok,None,2026-06-19T15:15:32.768384+00:00
87,BOE-A-2023-10306,20230428_BOE-A-2023-10306,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",Instalaciones eléctricas,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-A-2023-10306 Estatal Ministerio para la Tr...,33226,ok,None,2026-06-15T10:38:10.041511+00:00
124,BOE-B-2023-19082,20230622_BOE-B-2023-19082,https://www.boe.es/diario_boe/txt.php?id=BOE-B...,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,None,MINISTERIO DE POLÍTICA TERRITORIAL,V. Anuncios. - B. Otros anuncios oficiales,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-B-2023-19082 Ministerio de Política Territ...,12209,ok,None,2026-06-19T15:15:32.768384+00:00
165,BOE-A-2024-16664,20240810_BOE-A-2024-16664,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...",Instalaciones eléctricas,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-A-2024-16664 Estatal Ministerio para la Tr...,52234,ok,None,2026-06-19T15:15:32.768384+00:00
